# Expanded Heart Disease Data Analysis - SKELETON NOTEBOOK

## Practice Notebook (Fill in the Code)

**Objective:** Expand your skills in statistical hypothesis testing, visualization, and audience-aware data communication using the heart disease dataset.

**What you will practice:**
- Data inspection and EDA tailored to different audience data literacy levels (from the "What to Consider When Considering the Audience" guidelines)
- Hypothesis tests for quantitative and categorical predictors (t-test, ANOVA, Chi-square, and alternates)
- Effect sizes, multiple testing correction
- Logistic regression for combined predictor effects
- Simulation & sensitivity analysis (bootstrap, permutation tests)
- Creating a structured data analysis report (inspired by "Structure of a Data Analysis Report")
- **Flowchart** of the complete workflow included below

**How to use this skeleton:**
1. Read each markdown instruction carefully.
2. Fill in the `# YOUR CODE HERE` sections in code cells.
3. Run cells to see outputs and debug.
4. When stuck or to compare approaches, open the **solution notebook**.
5. Think: "Who is my audience for this result? High data literacy (clinicians, data scientists) or low (general public, executives)?"

**Dataset Variables (for reference):**
- `age`, `sex`, `trestbps` (resting BP), `chol` (cholesterol), `cp` (chest pain type: 4 levels), `exang` (exercise angina 0/1), `fbs` (fasting BS >120 0/1), `thalach` (max HR), `heart_disease` (presence/absence)

**Libraries you will need (already in first code cell).**

## 1. Setup and Library Imports

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical libraries
from scipy.stats import (ttest_ind, f_oneway, chi2_contingency, 
                         mannwhitneyu, bootstrap, permutation_test)
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from statsmodels.formula.api import logit

# Visualization settings for audience accessibility (colorblind friendly)
sns.set_palette('colorblind')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully. Ready for analysis!")

## 2. Flowchart of Desired Analysis Outcome & Workflow

This flowchart illustrates the **recommended end-to-end process** for producing rigorous, audience-aware insights from data. It incorporates:
- Data literacy & subject knowledge considerations (from audience analysis guidelines)
- Statistical best practices + alternates
- Simulation for robustness
- Structured reporting for different audiences (primary collaborator, executive skimmer, technical supervisor)

**Study this flowchart before starting.** It shows the "desired outcome" of a professional analysis.

In [ ]:
# Generate and display the analysis workflow flowchart
def create_analysis_flowchart():
    fig, ax = plt.subplots(figsize=(11, 15))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 17)
    ax.axis('off')
    
    # Define workflow steps with y-positions (top to bottom), text, and colors
    steps = [
        (5, 16.0, "1. LOAD & INSPECT DATA
• pd.read_csv, .head(), .info(), .describe()
• Check missing values, dtypes, distributions
• Consider audience: data literacy level & subject knowledge", "#BBDEFB"),
        (5, 14.2, "2. EDA & VISUALIZATIONS (Audience-Adapted)
• Boxplots, countplots, heatmaps, pairplots
• Use colorblind palettes, clear labels, annotations
• Simple bars for low-literacy audiences; advanced stats for experts", "#C8E6C9"),
        (5, 12.4, "3. UNIVARIATE HYPOTHESIS TESTS
• Quantitative vs binary (t-test / Mann-Whitney U + effect size)
• Multi-level vs quantitative (ANOVA / Kruskal-Wallis + post-hoc)
• Categorical vs binary (Chi-square / Fisher's Exact)", "#FFE0B2"),
        (5, 10.6, "4. MULTIPLE TESTING & POST-HOC
• Tukey HSD for pairwise comparisons
• Bonferroni / FDR correction across all tests
• Control family-wise error rate", "#FFCCBC"),
        (5, 8.8, "5. MULTIVARIABLE MODELING
• Logistic Regression (statsmodels) for combined effects
• Odds ratios, p-values, confidence intervals
• Interpretation for subject-matter experts (clinicians)", "#D1C4E9"),
        (5, 7.0, "6. SIMULATION & SENSITIVITY ANALYSIS
• Bootstrap confidence intervals for effects
• Permutation tests (non-parametric alternate)
• Monte Carlo power analysis; vary alpha, n, effect size", "#B2DFDB"),
        (5, 5.2, "7. TAILORED INSIGHTS & REPORTING
• Structure: Intro (questions) → Body (key evidence) → Conclusion → Appendix (details)
• Adapt language: executives (headlines), technical (methods), general (plain language)
• Visuals + narrative matched to audience needs", "#FFECB3"),
    ]
    
    for x, y, text, color in steps:
        # Rounded box
        box = FancyBboxPatch((x - 4.2, y - 0.85), 8.4, 1.7,
                             boxstyle="round,pad=0.03,rounding_size=0.15",
                             facecolor=color, edgecolor='#37474F', linewidth=2.0, alpha=0.95)
        ax.add_patch(box)
        ax.text(x, y, text, ha='center', va='center', fontsize=8.5,
                fontweight='medium', wrap=True, color='#212121',
                bbox=dict(boxstyle='square,pad=0', facecolor='none', edgecolor='none'))
    
    # Draw arrows between consecutive steps
    arrow_style = dict(arrowstyle='->', color='#455A64', lw=2.5, mutation_scale=15)
    for i in range(len(steps) - 1):
        y_start = steps[i][1] - 0.9
        y_end = steps[i + 1][1] + 0.95
        ax.annotate('', xy=(5, y_end), xytext=(5, y_start), arrowprops=arrow_style)
    
    # Title
    ax.text(5, 16.8, "HEART DISEASE DATA ANALYSIS WORKFLOW", ha='center', va='bottom',
            fontsize=13, fontweight='bold', color='#1565C0')
    ax.text(5, 16.5, "(Audience-Aware • Statistically Rigorous • Simulation-Validated)", 
            ha='center', va='top', fontsize=9, style='italic', color='#455A64')
    
    # Footer note
    ax.text(5, 0.3, "Inspired by audience analysis best practices & structured data analysis reporting guidelines",
            ha='center', va='bottom', fontsize=7, style='italic', color='#78909C')
    
    plt.tight_layout()
    plt.savefig(f"{'/home/workdir/artifacts'}/analysis_flowchart.png", dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print("Flowchart saved to analysis_flowchart.png")

create_analysis_flowchart()

## 3. Data Loading and Initial Inspection

**Task 3.1 (Data Literacy Check):** 
Load the dataset from `/home/workdir/attachments/hearth_disease.csv`. 
Use `.head()`, `.info()`, `.describe(include='all')`, and check for missing values and unique values in categorical columns (`sex`, `cp`, `heart_disease`, `exang`, `fbs`).

**Audience Consideration:** 
- For high data-literacy audiences (data scientists, cardiologists): show technical summaries, distributions, correlations.
- For low data-literacy (patients, hospital admins): focus on plain-language variable explanations and simple counts first.

Print the first 5 rows and basic info. Note the shape and any issues.

In [ ]:
# YOUR CODE HERE
# Load data
heart = pd.read_csv('/home/workdir/attachments/hearth_disease.csv')

# Inspect
print("=== First 5 rows ===")
print(heart.head())

print("\n=== Data Info ===")
print(heart.info())

print("\n=== Descriptive Statistics ===")
print(heart.describe(include='all'))

print("\n=== Missing Values ===")
print(heart.isnull().sum())

print("\n=== Categorical Value Counts ===")
for col in ['sex', 'cp', 'heart_disease', 'exang', 'fbs']:
    print(f"\n{col}:")
    print(heart[col].value_counts())

## 4. Exploratory Data Analysis & Visualizations (Audience-Adapted)

**Task 4.1:** Create side-by-side boxplots of `thalach` by `heart_disease`. 
Add title, labels, and mean annotations. Use colorblind palette (already set).

**Audience Tip:** For general audiences, add a short caption explaining "Patients without heart disease tend to achieve higher maximum heart rates during exercise tests."

**Task 4.2:** Create countplots or barplots for `cp` (chest pain) and `heart_disease`, and for `sex` vs `heart_disease`. 
Keep charts simple and well-labeled.

**Task 4.3 (Advanced for high-literacy audience):** Create a correlation heatmap of the quantitative variables. 
Use `sns.heatmap` with annotations. Note: avoid for low-literacy audiences as it can be overwhelming.

In [ ]:
# YOUR CODE HERE - Task 4.1: thalach by heart_disease boxplot
plt.figure(figsize=(8, 5))
# sns.boxplot(...)

# Add mean annotations (hint: use plt.text or sns.pointplot overlay)
plt.title('Maximum Heart Rate (thalach) by Heart Disease Diagnosis')
plt.xlabel('Heart Disease')
plt.ylabel('Maximum Heart Rate Achieved')
plt.show()

# YOUR CODE HERE - Task 4.2: Simple countplots for categorical relationships
# Example for cp
plt.figure(figsize=(10, 5))
# sns.countplot(data=heart, x='cp', hue='heart_disease')
plt.title('Chest Pain Type Distribution by Heart Disease Status')
plt.xticks(rotation=15)
plt.show()

# YOUR CODE HERE - Task 4.3 (optional advanced): Correlation heatmap
# quant_vars = ['age', 'trestbps', 'chol', 'thalach']
# corr = heart[quant_vars].corr()
# sns.heatmap(corr, annot=True, cmap='RdYlBu_r', center=0)
# plt.title('Correlation Heatmap of Quantitative Variables')
# plt.show()

## 5. Univariate Hypothesis Tests for Predictors of Heart Disease

**Task 5.1 - thalach (original + expanded):**
- Save `thalach_hd` and `thalach_no_hd`
- Calculate and print mean and median differences
- Run two-sample t-test (null: equal means)
- **Alternate method:** Run Mann-Whitney U test (non-parametric, does not assume normality)
- Compute Cohen's d effect size manually or via formula
- Print p-value and interpret at α=0.05. Is there evidence of association?

Repeat similar process for `age`, `trestbps`, `chol` (use `plt.clf()` before new plots).

**Task 5.2 - Chest Pain (cp) and thalach:**
- Boxplot of `thalach` by `cp` (4 categories)
- Run one-way ANOVA
- If significant, run Tukey's HSD post-hoc test
- **Alternate:** Kruskal-Wallis H-test + Dunn post-hoc (if needed)

**Task 5.3 - Categorical predictors vs heart_disease:**
Create contingency tables and run Chi-square tests for:
- `sex` vs `heart_disease`
- `exang` vs `heart_disease`
- `fbs` vs `heart_disease`
- `cp` vs `heart_disease` (already in original)

For 2x2 tables, **alternate method:** Fisher's Exact test (more accurate for small counts).

In [ ]:
# YOUR CODE HERE - Task 5.1 thalach analysis
thalach_hd = heart.thalach[heart.heart_disease == 'presence']
thalach_no_hd = heart.thalach[heart.heart_disease == 'absence']

mean_diff = np.mean(thalach_no_hd) - np.mean(thalach_hd)
med_diff = np.median(thalach_no_hd) - np.median(thalach_hd)
print(f"Mean difference (no HD - HD): {mean_diff:.3f}")
print(f"Median difference (no HD - HD): {med_diff:.3f}")

# t-test
tstat, pval_t = ttest_ind(thalach_hd, thalach_no_hd)
print(f"t-test p-value: {pval_t:.4e}")

# ALTERNATE: Mann-Whitney U
# u_stat, pval_u = mannwhitneyu(thalach_hd, thalach_no_hd)
# print(f"Mann-Whitney U p-value: {pval_u:.4e}")

# Cohen's d (pooled std)
# pooled_std = np.sqrt(((len(thalach_hd)-1)*thalach_hd.var() + (len(thalach_no_hd)-1)*thalach_no_hd.var()) / (len(thalach_hd) + len(thalach_no_hd) - 2))
# cohens_d = mean_diff / pooled_std
# print(f"Cohen's d effect size: {cohens_d:.3f}")

# Repeat for age, trestbps, chol ... (use plt.clf() before each new sns.boxplot)

## 6. Multiple Testing Correction & Post-Hoc Analysis

After running several tests, collect all p-values and apply correction to control false positives.

**Task 6.1:** 
- Collect p-values from the tests you ran (t-tests for 4 quantitative vars, Chi2 for 4 categorical, ANOVA for cp)
- Use `multipletests` from statsmodels with method='bonferroni' or 'fdr_bh'
- Print original vs corrected p-values and which remain significant.

In [ ]:
# YOUR CODE HERE
# Example p-value list (replace with your actual pvals)
pvals = [0.0001, 0.03, 0.12, 0.45, 0.001, 0.008]  # placeholder
# corrected = multipletests(pvals, method='bonferroni')
# print(corrected[1])  # corrected p-values

## 7. Multivariable Modeling: Logistic Regression

**Task 7.1:** 
Fit a logistic regression model to predict `heart_disease` (encode as binary: presence=1) using key predictors: age, thalach, chol, trestbps, and cp (as categorical).

Use `statsmodels.formula.api.logit` or `sm.Logit`.

Print the summary. Interpret coefficients as odds ratios (exp(beta)) for subject-matter experts (cardiologists).

**Audience Note:** For executives/nonspecialists, translate: "For every 10 bpm increase in max heart rate, the odds of heart disease decrease by X% (holding other factors constant)."

**Alternate method:** You can also use `sklearn.linear_model.LogisticRegression` for prediction-focused work, but statsmodels gives better statistical inference.

In [ ]:
# YOUR CODE HERE
# Prepare binary outcome
heart['hd_binary'] = (heart.heart_disease == 'presence').astype(int)

# Example formula (expand with more terms if desired)
# formula = 'hd_binary ~ age + thalach + chol + C(cp) + C(sex) + exang'
# model = logit(formula, data=heart).fit()
# print(model.summary())

# Odds ratios
# ors = np.exp(model.params)
# print("\nOdds Ratios:")
# print(ors)

# For prediction on new data or marginal effects, see solution notebook

## 8. Simulation & Sensitivity Analysis (Practice Modifying Values)

This section lets you experiment: change parameters and observe how results change. This builds intuition about statistical robustness.

**Task 8.1 - Bootstrap CI for mean difference (thalach):**
Use `scipy.stats.bootstrap` to get 95% CI for the difference in means. Compare to t-test CI.

**Task 8.2 - Permutation Test:**
Implement or use `permutation_test` as a fully non-parametric alternate to t-test. Does conclusion change?

**Task 8.3 - Sensitivity:**
- Change `alpha` from 0.05 to 0.01 or 0.10 and re-evaluate which findings remain significant after correction.
- Subsample the data (e.g., n=100 random rows) and re-run key tests. How stable are p-values?
- **Modify these values at the top of the cell and re-run:**
  ```python
  ALPHA = 0.05
  N_BOOT = 10000
  SUBSAMPLE_SIZE = None  # or e.g. 150
  ```

**Task 8.4 (Advanced):** Simple Monte Carlo power simulation - generate synthetic data with known effect size, run test many times, see % of significant results.

In [ ]:
# YOUR CODE HERE - Simulation parameters (MODIFY THESE)
ALPHA = 0.05
N_BOOT = 5000
SUBSAMPLE_SIZE = None  # e.g. 100 to test stability

print(f"Running simulations with alpha={ALPHA}, bootstrap reps={N_BOOT}")

# Bootstrap example for thalach mean diff CI
def mean_diff_stat(x, y):
    return np.mean(x) - np.mean(y)

# data1 = thalach_no_hd.values
# data2 = thalach_hd.values
# res = bootstrap((data1, data2), mean_diff_stat, n_resamples=N_BOOT, vectorized=False, method='percentile')
# print(f"Bootstrap 95% CI for mean diff: {res.confidence_interval}")

# Permutation test (uncomment in solution style)
# perm_res = permutation_test((thalach_hd.values, thalach_no_hd.values), 
#                             lambda x, y: np.mean(x) - np.mean(y),
#                             n_resamples=5000, alternative='two-sided')
# print(f"Permutation test p-value: {perm_res.pvalue:.4e}")

# Quick sensitivity check: subsample
if SUBSAMPLE_SIZE:
    heart_sub = heart.sample(n=SUBSAMPLE_SIZE, random_state=42)
    # Re-run a t-test on subsample and compare p-value...
    print("Subsample analysis placeholder - see solution for full code")

## 9. More Practice & Exploration Tasks

1. Investigate `exang` (exercise-induced angina) as a predictor. Use appropriate test + visualization. Is it strongly associated?
2. Check for interaction: does the thalach difference by heart_disease vary by `sex`? (stratified boxplots or interaction term in logit)
3. Create a clean summary table of all significant predictors with effect sizes and corrected p-values (suitable for technical appendix).
4. **Audience exercise:** Write two short paragraphs summarizing the key finding about `thalach`:
   - One for a cardiologist (use medical terms, stats)
   - One for a hospital administrator or patient (plain language, actionable insight)

Put your code and text in the cells below.

In [ ]:
# YOUR CODE HERE - Practice tasks
# Task 1: exang analysis
# crosstab = pd.crosstab(heart.exang, heart.heart_disease)
# chi2, p, _, _ = chi2_contingency(crosstab)
# print("exang vs heart_disease chi2 p-value:", p)

# Task 2: Stratified by sex (example)
# sns.boxplot(data=heart, x='heart_disease', y='thalach', hue='sex')
# plt.show()

# Task 4: Two summaries (text)
# specialist_summary = "Among patients with heart disease..."
# general_summary = "People who could not reach high heart rates during exercise testing were more likely to..." 

## 10. Conclusion & Next Steps (Your Turn)

Summarize your findings in a structured way:
- **Big questions answered:** Which variables are significantly associated with heart disease?
- **Key effect sizes and practical meaning**
- **Limitations and further work** (e.g. more variables, causal inference, external validation)

**Report Structure Reminder (for your final write-up):**
1. Introduction (questions + data summary)
2. Body (methods brief, results with visuals/tables for skimmers)
3. Conclusion (headlines + implications)
4. Appendix (detailed stats, code, extra figures)

When finished, compare your work and narrative to the solution notebook, which also includes full alternate code paths and a ready-to-adapt executive summary.